# 🚀 CIFAR-100 Training on Colab
自動掛載 Google Drive、clone repo、偵測 checkpoint 並恢復訓練。

## Step 1｜掛載 Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'✅ Checkpoint 目錄：{CHECKPOINT_DIR}')

!git config --global user.name "rvgoing"
!git config --global user.email "cmkdkimo@gmail.com"

Mounted at /content/drive
✅ Checkpoint 目錄：/content/drive/MyDrive/cifar100_checkpoints


## Step 2｜Clone / Pull 最新程式碼

In [4]:
GITHUB_REPO = 'https://github.com/rvgoing/CFIR_100_ReFresh.git'
REPO_NAME   = 'CFIR_100_ReFresh'

if os.path.exists(f'/content/{REPO_NAME}'):
    print('📦 Repo 已存在，執行 git pull ...')
    %cd /content/{REPO_NAME}
    !git pull
else:
    print('📦 Clone repo ...')
    !git clone {GITHUB_REPO}
    %cd /content/{REPO_NAME}

print('✅ 程式碼已是最新版本')

📦 Clone repo ...
Cloning into 'CFIR_100_ReFresh'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 36 (delta 15), reused 29 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 19.36 KiB | 6.45 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/CFIR_100_ReFresh
✅ 程式碼已是最新版本


## Step 3｜安裝依賴

In [5]:
!pip install -q -r requirements.txt
print('✅ 套件安裝完成')

✅ 套件安裝完成


## Step 4｜確認 GPU

In [6]:
import torch
print(f'PyTorch 版本：{torch.__version__}')
print(f'CUDA 可用：{torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU：{torch.cuda.get_device_name(0)}')
    print(f'VRAM：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
!nvidia-smi

PyTorch 版本：2.10.0+cu128
CUDA 可用：True
GPU：Tesla T4
VRAM：14.6 GB
Sat Apr 25 07:23:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |         

## Step 5｜自動偵測 Checkpoint 並開始訓練

In [12]:
import os

# 注意：副檔名是 .pth
BEST_CKPT = os.path.join(CHECKPOINT_DIR, 'model_best.pth')
LAST_CKPT = os.path.join(CHECKPOINT_DIR, 'checkpoint.pth')

if os.path.exists(LAST_CKPT):
    resume_path = LAST_CKPT
    print(f'🔄 發現 checkpoint，將從上次進度恢復：{resume_path}')
elif os.path.exists(BEST_CKPT):
    resume_path = BEST_CKPT
    print(f'🔄 發現 best checkpoint，將從此恢復：{resume_path}')
else:
    resume_path = ''
    print('🆕 未發現 checkpoint，從頭開始訓練')

resume_arg = f'--resume {resume_path}' if resume_path else ''

cmd = f"python train.py --epochs 200 --batch-size 128 --lr 0.1 --save-dir {CHECKPOINT_DIR} {resume_arg}"

print(f'\n🚀 執行指令：\n{cmd}\n')
!{cmd}

🆕 未發現 checkpoint，從頭開始訓練

🚀 執行指令：
python train.py --epochs 20 --batch-size 128 --lr 0.1 --save-dir /content/drive/MyDrive/cifar100_checkpoints 

Using device: cuda
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Epoch [1/20]
Training:   0% 0/391 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker cr

## Step 6｜確認 Checkpoint 已存到 Drive

In [14]:
import os

print('📂 Drive checkpoint 目錄內容：')
for f in sorted(os.listdir(CHECKPOINT_DIR)):
    size = os.path.getsize(os.path.join(CHECKPOINT_DIR, f)) / 1024**2
    print(f'  {f}  ({size:.1f} MB)')

📂 Drive checkpoint 目錄內容：
  checkpoint.pth  (85.7 MB)
  eval_results  (0.0 MB)
  model_best.pth  (85.7 MB)
  training_log.json  (0.0 MB)


## Step 7｜執行評量（訓練完成後執行）

In [15]:
EVAL_DIR = os.path.join(CHECKPOINT_DIR, 'eval_results')
os.makedirs(EVAL_DIR, exist_ok=True)

cmd = f"python evaluate.py --checkpoint {BEST_CKPT} --save-dir {EVAL_DIR}"

print(f'🔍 執行評量：\n{cmd}\n')
!{cmd}

print('\n📂 評量結果：')
for f in sorted(os.listdir(EVAL_DIR)):
    size = os.path.getsize(os.path.join(EVAL_DIR, f)) / 1024**2
    print(f'  {f}  ({size:.1f} MB)')

🔍 執行評量：
python evaluate.py --checkpoint /content/drive/MyDrive/cifar100_checkpoints/model_best.pth --save-dir /content/drive/MyDrive/cifar100_checkpoints/eval_results


========== CIFAR-100 Evaluation ==========
[1/5] Loading model ...
Loading checkpoint: /content/drive/MyDrive/cifar100_checkpoints/model_best.pth
  Loaded from epoch 18, best_acc=56.45%
[2/5] Running inference ...
[3/5] Computing metrics ...
      Top-1 Accuracy : 56.45%
      Top-5 Accuracy : 84.68%
[4/5] Saving plots ...
  Saved: /content/drive/MyDrive/cifar100_checkpoints/eval_results/loss_acc_curve.png
  Saved: /content/drive/MyDrive/cifar100_checkpoints/eval_results/confusion_matrix.png
  Saved: /content/drive/MyDrive/cifar100_checkpoints/eval_results/per_class_accuracy.png
[5/5] Saving reports ...
  Saved: /content/drive/MyDrive/cifar100_checkpoints/eval_results/classification_report.txt
  Saved: /content/drive/MyDrive/cifar100_checkpoints/eval_results/summary.json
  Saved: labels.npy / preds.npy

========== Summa

## Step 8｜繪製圖形（訓練完成後執行）

In [13]:
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
from evaluate import CIFAR100_CLASSES

# 直接讀已存好的結果，不需要重跑模型
labels = np.load(f'{EVAL_DIR}/labels.npy')
preds  = np.load(f'{EVAL_DIR}/preds.npy')

cm = confusion_matrix(labels, preds)
df_cm = pd.DataFrame(cm, index=CIFAR100_CLASSES, columns=CIFAR100_CLASSES)

errors   = cm.sum(axis=1) - cm.diagonal()
top_idx  = np.argsort(errors)[-15:][::-1]
top_names = [CIFAR100_CLASSES[i] for i in top_idx]
df_top   = df_cm.loc[top_names, top_names]

display(df_top.style
    .background_gradient(cmap='Blues')
    .format("{:d}")
    .set_caption("Confusion Matrix — Top 15 Most Confused Classes")
)

,lizard,bear,shrew,turtle,beaver,boy,poppy,girl,bowl,kangaroo,squirrel,crocodile,rabbit,wolf,man
lizard,2,0,1,1,0,0,0,0,0,0,0,5,0,0,0
bear,0,8,4,0,3,0,0,0,0,0,1,0,0,0,0
shrew,0,0,15,0,1,0,0,0,0,0,0,1,0,0,0
turtle,0,0,1,19,0,0,0,0,0,0,0,0,1,0,0
beaver,0,0,0,0,20,0,0,0,0,0,1,7,0,0,0
boy,0,0,0,0,0,23,0,17,0,0,0,0,0,0,10
poppy,0,0,0,0,0,0,24,0,0,0,0,0,1,0,0
girl,0,0,0,1,0,1,0,24,0,0,0,0,0,0,3
bowl,0,0,0,0,0,0,0,0,27,0,0,0,0,0,0
kangaroo,0,0,2,1,1,0,0,0,0,28,1,3,3,0,0


## Step 9｜清除 checkpoint

In [11]:
import os

CHECKPOINT_DIR = '/content/drive/MyDrive/cifar100_checkpoints'

for f in ['checkpoint.pth', 'model_best.pth', 'training_log.json']:
    path = os.path.join(CHECKPOINT_DIR, f)
    if os.path.exists(path):
        os.remove(path)
        print(f'已刪除：{f}')

print('✅ 完成，可以從頭訓練了')

已刪除：checkpoint.pth
已刪除：model_best.pth
✅ 完成，可以從頭訓練了
